# 02 · Monthly series and STL decomposition

**Question.** How do monthly recorded counts of each crime group evolve over 2022–2025: what is level, trend and seasonality, how strong is each component, and is there a monotonic trend?

**Methods and libraries.**
- **STL** — Seasonal-Trend decomposition using Loess (Cleveland, R.B., Cleveland, W.S., McRae, J.E. & Terpenning, I. 1990, *J. Official Statistics* 6(1)), `statsmodels.tsa.seasonal.STL` with `period=12`, `robust=True` (robust loess weights, so single outlying months do not bend trend/seasonal), statsmodels default windows (seasonal = 7, trend = next odd integer ≥ 1.5·period / (1 − 1.5/seasonal)).
- **Strength of trend / seasonality** — Wang, X., Smith, K. & Hyndman, R.J. (2006), *Data Mining and Knowledge Discovery* 13(3); Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* 3rd ed. (FPP3), §4.3. No statsmodels function returns these; the two formulas are written out below.
- **Mann–Kendall trend test + Sen's slope** — Mann (1945) *Econometrica* 13; Kendall (1975) *Rank Correlation Methods*; Sen (1968) *JASA* 63; `pymannkendall` (Hussain & Mahmud 2019, *JOSS* 4(39)). `original_test` (assumes independent months), `hamed_rao_modification_test` (variance corrected for autocorrelation; Hamed & Rao 1998, *J. Hydrology* 204) and `seasonal_test` with period 12 (Hirsch, Slack & Smith 1982, *Water Resources Research* 18(1); compares each calendar month only with itself, so seasonality cannot masquerade as trend).

**Data.** Two monthly series per group (48 months, 2022-01 … 2025-12, occurrence date):
1. **all bulletins** of the group, geocoded or not: `Warehouse.counts(CrimeQuery(crime_group=g, unit="city", time_unit="month"))` (the temporal step includes `sexual`, which has no coordinates); `all` = every bulletin in the window;
2. **in-tract** (spatial event set): `Warehouse.counts(CrimeQuery(crime_group=g, time_unit="month"))` summed over tracts.

STL, strengths and trend tests are run on both; the component plots show the all-bulletin series.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymannkendall as mk
from statsmodels.tsa.seasonal import STL

from crime.core.config import DATE_END, DATE_START, NOTEBOOK_OUTPUTS_DIR
from crime.data import SPATIAL_GROUPS as GROUPS
from crime.data import TEMPORAL_GROUPS, CrimeQuery, Warehouse

OUT = NOTEBOOK_OUTPUTS_DIR / "02_stl"
OUT.mkdir(parents=True, exist_ok=True)
PERIOD = 12          # months per seasonal cycle
ROBUST = True        # robust loess (Cleveland et al. 1990, §3)
ALPHA = 0.05

GROUP_COLORS = {"all": "#2a78d6", "property": "#eb6834", "bodily_harm": "#1baf7a",
                "traffic": "#eda100", "drug": "#e87ba4", "violent": "#008300", "sexual": "#6250d6"}
GROUP_LABEL = {"all": "All groups", "property": "Property", "bodily_harm": "Bodily harm",
               "traffic": "Traffic", "drug": "Drug", "violent": "Violent", "sexual": "Sexual"}
TEMPORAL_GROUPS = list(TEMPORAL_GROUPS)
wh = Warehouse()
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})

def save(fig, name):
    for ax in fig.axes:                       # one tick per year on time axes
        if isinstance(ax.xaxis.get_major_formatter(), (mdates.AutoDateFormatter, mdates.ConciseDateFormatter)):
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.savefig(OUT / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
months = pd.date_range(DATE_START, DATE_END, freq="MS")


def monthly(group, unit):
    """Bulletins of one group per month 2022-2025 (long table from the warehouse, summed over tracts)."""
    c = wh.counts(CrimeQuery(crime_group=group, unit=unit, time_unit="month"))
    return c.groupby("period_start")["count"].sum()


all_bul = pd.DataFrame({g: monthly(g, "city") for g in TEMPORAL_GROUPS}).reindex(months, fill_value=0)
in_tract = pd.DataFrame({g: monthly(g, "census_tract") for g in GROUPS})
in_tract.index = pd.DatetimeIndex(in_tract.index, name="period")
assert len(all_bul) == len(in_tract) == 48
assert (all_bul.drop(columns="all").sum(axis=1) == all_bul["all"]).all()   # the groups partition "all"
series = {"all_bulletins": all_bul, "in_tract": in_tract}
pd.concat(series, axis=1).to_csv(OUT / "monthly_series.csv")
print("bulletins in window:", int(all_bul["all"].sum()), "| in-tract:", int(in_tract["all"].sum()))
pd.DataFrame({"all_bulletins": all_bul.sum(), "in_tract": in_tract.sum()}).astype("Int64")

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(10, 8.5), sharex=True)
for ax, g in zip(axes.flat, TEMPORAL_GROUPS):
    ax.plot(all_bul.index, all_bul[g], color=GROUP_COLORS[g], lw=1.6, label="all bulletins")
    if g in in_tract:
        ax.plot(in_tract.index, in_tract[g], color=GROUP_COLORS[g], lw=1.2, ls="--", label="geocoded in tract")
    ax.set_ylim(0, all_bul[g].max() * 1.1)
    ax.set_title(GROUP_LABEL[g])
    ax.set_ylabel("Bulletins per month")
axes[0, 0].legend(frameon=False, fontsize=8)
axes.flat[-1].set_axis_off()
fig.suptitle("Monthly police bulletins by crime group: all vs geocoded in-tract (own y-scale per panel)")
fig.tight_layout()
save(fig, "monthly_all_vs_in_tract")

## STL decomposition

`STL(y, period=12, robust=True).fit()` gives y = T + S + R. The robustness weights (`res.weights`, 1 = ordinary, near 0 = downweighted) flag months the fit treats as outliers.

In [ ]:
fits = {}
for kind, df in series.items():
    for g in df.columns:
        fits[(kind, g)] = STL(df[g].astype(float), period=PERIOD, robust=ROBUST).fit()

comp = pd.concat({f"{k}|{g}": pd.DataFrame({"observed": r.observed, "trend": r.trend, "seasonal": r.seasonal,
                                            "resid": r.resid, "robust_weight": r.weights})
                  for (k, g), r in fits.items()}, axis=1)
comp.to_csv(OUT / "stl_components.csv")

for g in TEMPORAL_GROUPS:
    r = fits[("all_bulletins", g)]
    fig, axes = plt.subplots(4, 1, figsize=(8, 7), sharex=True)
    axes[0].plot(r.observed.index, r.observed, color=GROUP_COLORS[g], lw=1.5)
    axes[0].set_ylabel("Observed\n(bulletins / month)")
    axes[1].plot(r.trend.index, r.trend, color=GROUP_COLORS[g], lw=1.8)
    axes[1].set_ylabel("Trend T\n(bulletins / month)")
    axes[2].plot(r.seasonal.index, r.seasonal, color=GROUP_COLORS[g], lw=1.5)
    axes[2].axhline(0, color="#52514e", lw=0.8)
    axes[2].set_ylabel("Seasonal S\n(bulletins / month)")
    axes[3].vlines(r.resid.index, 0, r.resid, color=GROUP_COLORS[g], lw=1.5)
    low = r.weights < 0.5
    axes[3].scatter(r.resid.index[low], r.resid[low], s=18, facecolor="white", edgecolor="#0b0b0b",
                    zorder=3, label="robust weight < 0.5")
    axes[3].axhline(0, color="#52514e", lw=0.8)
    axes[3].set_ylabel("Remainder R\n(bulletins / month)")
    if low.any():
        axes[3].legend(frameon=False, fontsize=8)
    axes[3].set_xlabel("Month of occurrence")
    fig.suptitle(f"STL (period 12, robust) of monthly bulletins — {GROUP_LABEL[g]} (all bulletins, geocoded or not)")
    fig.tight_layout()
    save(fig, f"stl_{g}")

## Strength of trend and seasonality

Wang, Smith & Hyndman (2006); FPP3 §4.3 (no library function, so written out):

$$F_T = \max\!\left(0,\; 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(T_t + R_t)}\right), \qquad
F_S = \max\!\left(0,\; 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(S_t + R_t)}\right)$$

Both lie in [0, 1]; values near 1 mean the component dominates the remainder. Also reported: the seasonal amplitude (max − min of S, bulletins/month), the peak and trough calendar months of S, and the trend change from the first to the last month.

In [ ]:
def strengths(r):
    vR = np.var(r.resid, ddof=1)
    return (max(0.0, 1 - vR / np.var(r.trend + r.resid, ddof=1)),
            max(0.0, 1 - vR / np.var(r.seasonal + r.resid, ddof=1)))

rows = []
for (kind, g), r in fits.items():
    ft, fs = strengths(r)
    s_mean = r.seasonal.groupby(r.seasonal.index.month).mean()
    rows.append({"series": kind, "group": g, "mean_per_month": r.observed.mean(),
                 "trend_strength_FT": ft, "seasonal_strength_FS": fs,
                 "seasonal_amplitude": r.seasonal.max() - r.seasonal.min(),
                 "seasonal_peak_month": int(s_mean.idxmax()), "seasonal_trough_month": int(s_mean.idxmin()),
                 "trend_first": r.trend.iloc[0], "trend_last": r.trend.iloc[-1],
                 "trend_change_%": 100 * (r.trend.iloc[-1] / r.trend.iloc[0] - 1),
                 "months_robust_weight_lt_0.5": int((r.weights < 0.5).sum())})
strength = pd.DataFrame(rows)
strength.to_csv(OUT / "stl_strength.csv", index=False)
strength.round(2)

In [ ]:
s = strength[strength["series"] == "all_bulletins"].set_index("group")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for g, row in s.iterrows():
    ax.scatter(row["seasonal_strength_FS"], row["trend_strength_FT"], s=60, color=GROUP_COLORS[g],
               edgecolor="white", zorder=3)
    ax.annotate(GROUP_LABEL[g], (row["seasonal_strength_FS"], row["trend_strength_FT"]),
                xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Seasonal strength $F_S$ (0 = none, 1 = dominates remainder)")
ax.set_ylabel("Trend strength $F_T$ (0 = none, 1 = dominates remainder)")
ax.set_title("STL component strength, monthly bulletins 2022–2025\n(all bulletins per group; Wang, Smith & Hyndman 2006)")
save(fig, "stl_strength")

## Sensitivity: robust vs non-robust STL on a 4-cycle series

With only 48 months each calendar-month sub-series has 4 values, and statsmodels' default seasonal window (7) spans all of them. The robust iterations can then down-weight the middle years to almost zero, after which the seasonal loess passes (almost) exactly through the first and last year: the remainder becomes ~0 in 2022 and 2025 and all irregular variation is pushed into 2023–2024 (visible in the plots above). The table counts months whose remainder is below 1 bulletin in absolute value and compares the strengths with an ordinary (`robust=False`) STL, same windows.

In [ ]:
rows = []
for (kind, g), r in list(fits.items()):
    r0 = STL(series[kind][g].astype(float), period=PERIOD, robust=False).fit()
    ft, fs = strengths(r)
    ft0, fs0 = strengths(r0)
    rows.append({"series": kind, "group": g,
                 "robust_months_|R|<1": int((r.resid.abs() < 1).sum()),
                 "robust_months_weight<0.5": int((r.weights < 0.5).sum()),
                 "nonrobust_months_|R|<1": int((r0.resid.abs() < 1).sum()),
                 "FT_robust": ft, "FT_nonrobust": ft0, "FS_robust": fs, "FS_nonrobust": fs0,
                 "trend_change_%_robust": 100 * (r.trend.iloc[-1] / r.trend.iloc[0] - 1),
                 "trend_change_%_nonrobust": 100 * (r0.trend.iloc[-1] / r0.trend.iloc[0] - 1)})
    fits[(kind + "_nonrobust", g)] = r0
sens = pd.DataFrame(rows)
sens.to_csv(OUT / "stl_robust_vs_nonrobust.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(10, 5.5), sharex=True)
for j, g in enumerate(["all", "property"]):
    r, r0 = fits[("all_bulletins", g)], fits[("all_bulletins_nonrobust", g)]
    axes[0, j].plot(r.observed.index, r.observed, color="#b8b8b2", lw=1, label="observed")
    axes[0, j].plot(r.trend.index, r.trend, color=GROUP_COLORS[g], lw=1.8, label="trend, robust")
    axes[0, j].plot(r0.trend.index, r0.trend, color=GROUP_COLORS[g], lw=1.4, ls="--", label="trend, non-robust")
    axes[0, j].set_title(GROUP_LABEL[g] + " (all bulletins)")
    axes[0, j].set_ylabel("Bulletins per month")
    axes[1, j].vlines(r.resid.index, 0, r.resid, color=GROUP_COLORS[g], lw=1.8, label="robust")
    axes[1, j].plot(r0.resid.index, r0.resid, "o", ms=3, color="#0b0b0b", label="non-robust")
    axes[1, j].axhline(0, color="#52514e", lw=0.8)
    axes[1, j].set_ylabel("Remainder R (bulletins / month)")
    axes[1, j].set_xlabel("Month of occurrence")
axes[0, 0].legend(frameon=False, fontsize=7); axes[1, 0].legend(frameon=False, fontsize=7)
fig.suptitle("STL with and without robust weights (period 12, default windows, 48 months)")
fig.tight_layout()
save(fig, "stl_robust_vs_nonrobust")
sens.round(2)

## Mann–Kendall trend test and Sen's slope

On each monthly series (48 values): `pymannkendall.original_test`, `hamed_rao_modification_test` and `seasonal_test(period=12)`. Sen's slope is the median of pairwise slopes, in bulletins per month per month (×12 → per year). The seasonal test's slope (pymannkendall `seasonal_sens_slope`) is the median of slopes between the same calendar month in different years, i.e. the yearly change of one month's count, reported in its own column.

In [ ]:
rows = []
for kind, df in series.items():
    for g in df.columns:
        y = df[g].to_numpy(dtype=float)
        o = mk.original_test(y, alpha=ALPHA)
        h = mk.hamed_rao_modification_test(y, alpha=ALPHA)
        sz = mk.seasonal_test(y, period=PERIOD, alpha=ALPHA)
        rows.append({"series": kind, "group": g,
                     "mk_trend": o.trend, "mk_tau": o.Tau, "mk_z": o.z, "mk_p": o.p,
                     "sen_slope_per_month": o.slope, "sen_slope_per_year": 12 * o.slope,
                     "sen_slope_%_of_mean_per_year": 100 * 12 * o.slope / y.mean(),
                     "hamed_rao_z": h.z, "hamed_rao_p": h.p, "hamed_rao_trend": h.trend,
                     "seasonal_mk_z": sz.z, "seasonal_mk_p": sz.p, "seasonal_mk_trend": sz.trend,
                     "seasonal_sen_slope_per_year": sz.slope})
mkt = pd.DataFrame(rows)
mkt.to_csv(OUT / "mann_kendall_sen.csv", index=False)
mkt.round(4)

In [ ]:
d = mkt[mkt["series"] == "all_bulletins"].set_index("group").loc[TEMPORAL_GROUPS]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
yy = np.arange(len(d))[::-1]
ax.barh(yy, d["sen_slope_%_of_mean_per_year"], color=[GROUP_COLORS[g] for g in d.index], height=0.6)
for yi, (g, row) in zip(yy, d.iterrows()):
    star = "significant" if row["seasonal_mk_p"] < ALPHA else "n.s."
    v = row["sen_slope_%_of_mean_per_year"]
    pv = row["seasonal_mk_p"]
    ptxt = "p < 0.001" if pv < 0.001 else f"p = {pv:.3f}"
    ax.text(max(v, 0) + 1, yi, f"{v:+.1f}%/yr (seasonal MK {ptxt}, {star})", va="center", ha="left", fontsize=7)
ax.axvline(0, color="#52514e", lw=0.8)
ax.set_yticks(yy, [GROUP_LABEL[g] for g in d.index])
lo = min(d["sen_slope_%_of_mean_per_year"].min(), 0) * 1.3
ax.set_xlim(lo, d["sen_slope_%_of_mean_per_year"].max() * 3.2)
ax.set_xlabel("Sen's slope, % of the group's mean monthly count per year")
ax.set_title("Monotonic trend of monthly bulletins 2022–2025 (all bulletins per group)")
ax.grid(axis="y", visible=False)
save(fig, "sen_slope_by_group")

## Reporting lag check

Time is the occurrence date; bulletins registered after the data extract are missing. The registration delay (`DATA_REGISTRO − DATA_OCORRENCIA_BO`) of the bulletins we do have shows how much of a month is usually registered late.

In [ ]:
bb = wh.bulletins(CrimeQuery(unit="city"))          # every bulletin in the window, geocoded or not
delay = (bb["registration_date"] - bb["occurrence_date"]).dt.days
lag = pd.Series({"median_delay_days": delay.median(), "share_registered_>30d_%": 100 * (delay > 30).mean(),
                 "share_registered_>90d_%": 100 * (delay > 90).mean(),
                 "latest_registration": bb["registration_date"].max().date()})
lag.to_frame("value")

## Results pending corrected execution

Regenerate this section from the corrected output tables. Historical narrative has been removed because it disagreed with executed output or preceded changes to inference and missing-data handling.


In [ ]:
for kind in series:
    st = strength[strength["series"] == kind].set_index("group")
    m = mkt[mkt["series"] == kind].set_index("group")
    print(f"--- {kind}")
    for g in st.index:
        print(f"{g:12s} mean {st.loc[g,'mean_per_month']:7.1f}/mo  F_T {st.loc[g,'trend_strength_FT']:.2f}  "
              f"F_S {st.loc[g,'seasonal_strength_FS']:.2f}  trend {st.loc[g,'trend_change_%']:+6.1f}%  "
              f"S peak/trough month {st.loc[g,'seasonal_peak_month']}/{st.loc[g,'seasonal_trough_month']}  "
              f"Sen {m.loc[g,'sen_slope_per_year']:+7.1f}/yr ({m.loc[g,'sen_slope_%_of_mean_per_year']:+.1f}%)  "
              f"MK p {m.loc[g,'mk_p']:.4f}  HR p {m.loc[g,'hamed_rao_p']:.4f}  seasonal MK p {m.loc[g,'seasonal_mk_p']:.4f}")

## Results pending corrected execution

Regenerate this section from the corrected output tables. Historical narrative has been removed because it disagreed with executed output or preceded changes to inference and missing-data handling.
